# Epithelial reintegration, specimen origin and CNV sensitivity

## Setup

In [1]:
import gc
import hashlib
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp

import scanpy as sc
import anndata as ad
import infercnvpy as cnv
import scvi

import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.lines import Line2D

import rpy2.robjects as ro
import rpy2.rinterface_lib.callbacks as rcb
import logging

os.environ["RPY2_CFFI_MODE"] = "ABI"

In [2]:
np.random.seed(0)
scvi.settings.seed = 0
scvi.settings.num_workers = 4

sc.settings.verbosity = 2
rcb.logger.setLevel(logging.ERROR)


Seed set to 0


In [1]:
from cap.paths import (
    project_root,
    data_dir as get_data_dir,
    figures_dir as get_figures_dir,
    dataframe_dir as get_dataframe_dir,
    save_dir as get_save_dir,
    model_dir as get_model_dir,
    display_path, notebook_dir)

project = project_root()
data_dir = get_data_dir()
fig_dir = get_figures_dir()
df_dir = get_dataframe_dir()
save_dir = get_save_dir()
model_dir = get_model_dir()
notebooks_root = notebook_dir()

ImportError: cannot import name 'notebook' from 'cap.paths' (/Volumes/TOSHIBA EXT/IdeaProjects/Nowotny_publication/src/cap/paths.py)

In [3]:
from cap.figures import set_style, save_figure, CONDITION_PALETTE, CONDITION_ORDER, CELLTYPE_PALETTE, categorical_palette, sample_order_by_condition
set_style()

In [4]:
target_genes = ["ROR1", "ROR2", "EGFR", "ERBB2", "EPCAM"]


In [5]:
random_seed = 0
n_hvg_primary = 4000
cnv_high_quantile = 0.95
infercnv_dynamic_threshold = 1.5 #package default
cnv_quantiles = (0.90, 0.95, 0.99)


In [6]:
n_latent = 20
n_layers = 2
gene_likelihood = "nb"
n_neighbors = 30
primary_resolution = 0.5
leiden_resolutions = (0.5, 1.0, 1.2, 1.5)


In [9]:
for d in [fig_dir, df_dir, save_dir, model_dir]:
    d.mkdir(parents=True, exist_ok=True)


In [2]:
meta_programme_file = data_dir / "cancer_meta_programme_signatures.xlsx"
gencode_file = data_dir / "gencode.v50.annotation.gtf.gz"
source_tumour_file = (data_dir / "source_tumour_cluster_barcodes.csv.gz")


NameError: name 'fig_dir' is not defined

In [11]:
for path in [meta_programme_file, gencode_file, source_tumour_file]:
    assert path.exists(), path


AssertionError: /Volumes/TOSHIBA EXT/IdeaProjects/Nowotny_publication/Data/gencode.v50.annotation.gtf.gz

In [ ]:
source_tumour_sha256 = ("26af400ae38a5e878873c1cb30e83b8447440518a8727fdf7a8808d7780ede5c")
actual_source_tumour_sha256 = hashlib.sha256(
    source_tumour_file.read_bytes()
).hexdigest()
assert actual_source_tumour_sha256 == source_tumour_sha256


In [ ]:
def rel(path):
    return Path(path).relative_to(project)


## Load the epithelial compartment

In [ ]:
epi = sc.read_h5ad(save_dir / "epithelial.h5ad")


legacy_celltypist_columns = {
    "celltypist_pal_label": "celltypist_reference_label",
    "celltypist_pal_conf": "celltypist_reference_confidence",
}


In [ ]:
epi.obs.rename(
    columns={
        old: new for old, new in legacy_celltypist_columns.items()
        if old in epi.obs.columns and new not in epi.obs.columns
    },
    inplace=True,
)


In [ ]:
display(pd.DataFrame({
    "cells": [epi.n_obs],
    "genes": [epi.n_vars],
    "samples": [epi.obs["sample"].nunique()],
}))


In [ ]:
# Keep the atlas coordinates from 002 before calculating the epithelial embedding.
if "X_umap" in epi.obsm and "X_umap_atlas" not in epi.obsm:
    epi.obsm["X_umap_atlas"] = epi.obsm["X_umap"].copy()


In [ ]:
counts = epi.layers["counts"]
count_values = counts.data if sp.issparse(counts) else np.asarray(counts).ravel()

assert np.isfinite(count_values).all()
assert np.allclose(count_values, np.rint(count_values))

baseline_positive = {
    gene: int((np.asarray(epi[:, gene].layers["counts"].todense()).ravel() > 0).sum())
    for gene in target_genes
}
display(pd.Series(baseline_positive, name="cells_with_raw_count"))


## Gene filtering and normalisation

In [ ]:
n_cells_loaded = epi.n_obs
n_genes_loaded = epi.n_vars


In [ ]:
sc.pp.filter_genes(epi, min_cells=3)
n_genes_removed = n_genes_loaded - epi.n_vars


In [ ]:
n_cells_start = epi.n_obs
n_genes_start = epi.n_vars


In [ ]:
shape_summary = pd.DataFrame({
    "stage": ["loaded", "analysis object"],
    "cells": [n_cells_loaded, epi.n_obs],
    "genes": [n_genes_loaded, epi.n_vars],
})
display(shape_summary)


In [ ]:
assert "log1p" in epi.layers, "Expected log1p layer in epithelial.h5ad"
lognorm = epi.layers["log1p"]
del epi.layers["log1p"]
epi.layers["lognorm"] = lognorm
epi.X = lognorm
del lognorm


In [ ]:
# the master shape that must survive inferCNV, scVI and clustering unchanged
n_cells_start = epi.n_obs
n_genes_start = epi.n_vars

assert n_cells_start == n_cells_loaded
assert n_genes_start == n_genes_loaded - n_genes_removed


## Highly variable genes



Binomial deviance (`scry`) computed per sample and aggregated by the maximum
across samples, so that a gene deviant in a single condition is retained.
Mitochondrial, ribosomal and haemoglobin genes are excluded before ranking.
Cell-cycle genes are deliberately **not** excluded here: proliferation is a
tumour-associated epithelial state of interest.

In [ ]:
%load_ext rpy2.ipython


In [ ]:
%%R
suppressPackageStartupMessages({
    library(Matrix)
    library(SingleCellExperiment)
    library(scry)
})


In [ ]:
def scipy_to_dgc(matrix):
    # transfer a scipy sparse matrix to R as a dgCMatrix
    matrix = sp.csc_matrix(matrix)
    return ro.r["new"](
        "dgCMatrix",
        i=ro.IntVector(matrix.indices),
        p=ro.IntVector(matrix.indptr),
        x=ro.FloatVector(matrix.data),
        Dim=ro.IntVector(matrix.shape),
    )


In [ ]:
samples = epi.obs["sample"].astype(str).unique().tolist()

deviance = pd.DataFrame(index=epi.var_names, columns=samples, dtype=float)


In [ ]:
for sample in samples:

    sub = epi[epi.obs["sample"].astype(str) == sample]

    ro.globalenv["raw_counts"] = scipy_to_dgc(sub.layers["counts"].T)
    ro.globalenv["gene_names"] = ro.StrVector(sub.var_names.tolist())
    ro.globalenv["cell_names"] = ro.StrVector(sub.obs_names.tolist())

    ro.r('''
        rownames(raw_counts) <- gene_names
        colnames(raw_counts) <- cell_names

        sce <- SingleCellExperiment(assays = list(counts = raw_counts))
        sce <- devianceFeatureSelection(sce)

        binomial_deviance <- rowData(sce)$binomial_deviance
    ''')

    deviance[sample] = np.asarray(ro.r["binomial_deviance"])

    ro.r("rm(raw_counts, sce, binomial_deviance); gc()")

    print(f"{sample}: {sub.n_obs:,} cells")


In [ ]:
epi.var["binomial_deviance"] = deviance.max(axis=1)

assert epi.var["binomial_deviance"].notna().all()


In [ ]:
for column in ["mt", "ribo", "hb"]:
    assert column in epi.var.columns, column

exclude = epi.var[["mt", "ribo", "hb"]].any(axis=1)


In [ ]:
ranked = (
    epi.var.loc[~exclude, "binomial_deviance"]
    .sort_values(ascending=False)
    .index
)

deviant_genes = ranked[:n_hvg_primary]


In [ ]:
# These genes are tracked for a sensitivity model, but do not enter the primary model unless selected by deviance.
meta_programme_sheet = pd.read_excel(meta_programme_file, sheet_name="Cancer MPs")
meta_programme_genes = (
    meta_programme_sheet.stack().dropna().astype(str).str.strip().loc[lambda x: x.ne("")].unique()
)


In [ ]:
protected = sorted(set(target_genes) | set(meta_programme_genes))
protected_present = [gene for gene in protected if gene in epi.var_names]


In [ ]:
epi.var["highly_variable_deviance"] = epi.var_names.isin(deviant_genes)
epi.var["highly_variable_protected"] = (
    epi.var["highly_variable_deviance"] | epi.var_names.isin(protected_present)
)

# The primary embedding is independent of the antigen and meta-programme lists.
epi.var["highly_variable"] = epi.var["highly_variable_deviance"].copy()

n_hvg_deviance = int(epi.var["highly_variable_deviance"].sum())
n_hvg_protected = int(epi.var["highly_variable_protected"].sum())
n_protected_added = n_hvg_protected - n_hvg_deviance


In [ ]:
hvg_summary = pd.DataFrame({
    "feature_set": ["primary_deviance", "protected_sensitivity"],
    "n_genes": [n_hvg_deviance, n_hvg_protected],
})
display(hvg_summary)

pd.Series(deviant_genes, name="gene").to_csv(
    df_dir / "epithelial_hvg_genes.csv", index=False
)


## inferCNV


Run inferCNV on a separate object so its gene filtering cannot alter the integration object. Genomic coordinates come from GENCODE v50 and are joined by stable Ensembl gene ID. The primary run uses infercnvpy's documented default dynamic threshold of 1.5. The normal-reference quantile sensitivity below does not test this smoothing parameter.

In [ ]:
# inferCNV needs log-normalised expression, the sample annotation and the gene coordinates. A full epi.copy() would also carry the counts layer and double peak memory for no benefit, so the working object is built slim.
cnv_adata = ad.AnnData(
    X=epi.layers["lognorm"].copy(),
    obs=epi.obs.copy(),
    var=epi.var.copy(),
)

print(f"inferCNV working object: {cnv_adata.n_obs:,} cells x {cnv_adata.n_vars:,} genes")


In [ ]:
gtf = pd.read_csv(
    gencode_file,
    sep="\t",
    comment="#",
    header=None,
    names=["chromosome", "source", "feature", "start", "end",
           "score", "strand", "frame", "attribute"],
)


In [ ]:
gencode = gtf[gtf["feature"] == "gene"].copy()


In [ ]:
gencode["ensembl"] = (
    gencode["attribute"].str.extract(r'gene_id "([^"]+)"')[0].str.split(".").str[0]
)


In [ ]:
gencode["gene_type"] = gencode["attribute"].str.extract(r'gene_type "([^"]+)"')[0]


In [ ]:
gencode = gencode.drop_duplicates("ensembl").set_index("ensembl")


In [ ]:
len(gencode)


In [ ]:
del gtf
gc.collect()


In [ ]:
# join on stable Ensembl identifiers taken from the dataset's own annotation
ensembl = cnv_adata.var["gene_id"].astype(str).str.split(".").str[0]


In [ ]:
cnv_adata.var["chromosome"] = ensembl.map(gencode["chromosome"]).values
cnv_adata.var["start"] = ensembl.map(gencode["start"]).values
cnv_adata.var["end"] = ensembl.map(gencode["end"]).values
cnv_adata.var["gene_type"] = ensembl.map(gencode["gene_type"]).values

resolved = cnv_adata.var["chromosome"].notna()


In [ ]:
del gencode
gc.collect()


In [ ]:
resolved.sum(), cnv_adata.n_vars


In [ ]:
cnv_adata.var.loc[~resolved, ["gene_id"]].to_csv(df_dir / "infercnv_unresolved_genes.csv")


In [ ]:
# protein-coding autosomal genes only


In [ ]:
autosomes = [f"chr{i}" for i in range(1, 23)]

keep = (
    cnv_adata.var["chromosome"].isin(autosomes)
    & cnv_adata.var["gene_type"].eq("protein_coding")
)


In [ ]:
cnv_adata = cnv_adata[:, keep].copy()


In [ ]:
cnv_adata.n_vars


In [ ]:
# gene families whose variation is not driven by copy number
problematic = (
    cnv_adata.var_names.str.startswith(
        ("MT-", "RPS", "RPL", "HLA-", "IGH", "IGK", "IGL", "KRT", "SPRR", "S100A")
    )
    | cnv_adata.var_names.isin(["MALAT1", "NEAT1"])
)


In [ ]:
problematic.sum()


In [ ]:
cnv_adata = cnv_adata[:, ~problematic].copy()


In [ ]:
sc.pp.filter_genes(cnv_adata, min_cells=100)


In [ ]:
cnv_adata.n_vars


In [ ]:
# genomic ordering
cnv_adata.var["chromosome"] = cnv_adata.var["chromosome"].astype(str)
cnv_adata.var["chr_num"] = cnv_adata.var["chromosome"].str.replace("chr", "", regex=False).astype(int)

order = cnv_adata.var.sort_values(["chr_num", "start"]).index
cnv_adata = cnv_adata[:, order].copy()

assert cnv_adata.var_names.is_unique
assert (np.diff(cnv_adata.var["chr_num"].to_numpy()) >= 0).all()


In [ ]:
# normal epithelium provides the diploid reference
reference = cnv_adata[cnv_adata.obs["condition"] == "N"].copy()

print(f"reference cells: {reference.n_obs:,} from {reference.obs['sample'].nunique()} samples")


In [ ]:
cnv_adata.obs["cnv_score"] = np.nan
cnv_adata.obs["cnv_signal"] = np.nan

x_cnv_full = None
failures = []

cnv_samples = cnv_adata.obs["sample"].astype(str).unique().tolist()


In [ ]:
len(cnv_samples)


In [ ]:
for sample in cnv_samples:

    print(f"\n{sample}")

    target = refs = combined = None

    try:
        target = cnv_adata[cnv_adata.obs["sample"].astype(str) == sample].copy()
        condition = str(target.obs["condition"].iloc[0])

        # a normal sample is never its own reference
        if condition == "N":
            refs = reference[reference.obs["sample"].astype(str) != sample].copy()
        else:
            refs = reference.copy()

        assert refs.n_obs > 0

        refs.obs["cnv_reference"] = "reference"
        target.obs["cnv_reference"] = "query"

        combined = sc.concat([refs, target], join="inner", merge="same")

        print(f"  {condition}: {target.n_obs:,} query, {refs.n_obs:,} reference")

        # X holds log-normalised expression, so no layer is passed
        cnv.tl.infercnv(
            combined,
            reference_key="cnv_reference",
            reference_cat=["reference"],
            window_size=100,
            step=10,
            dynamic_threshold=infercnv_dynamic_threshold,
            exclude_chromosomes=("chrX", "chrY"),
            n_jobs=1,
        )

        x_cnv = combined.obsm["X_cnv"]
        x_cnv = x_cnv.toarray() if sp.issparse(x_cnv) else np.asarray(x_cnv)
        x_cnv = x_cnv.astype(np.float32)

        query = combined.obs["cnv_reference"].eq("query").to_numpy()
        query_names = combined.obs_names[query]
        x_query = x_cnv[query]

        # map each inferCNV window to its chromosome
        chr_pos = sorted(combined.uns["cnv"]["chr_pos"].items(), key=lambda kv: kv[1])
        n_windows = x_query.shape[1]

        chr_labels = np.empty(n_windows, dtype=object)

        for i, (chrom, start) in enumerate(chr_pos):
            end = chr_pos[i + 1][1] if i + 1 < len(chr_pos) else n_windows
            chr_labels[start:end] = chrom

        # Chromosome RMS burden weighted by window count; chromosomes with more windows receive more weight.
        chrom_scores = []
        chrom_weights = []

        for chrom in pd.unique(chr_labels):
            idx = np.flatnonzero(chr_labels == chrom)
            chrom_scores.append(np.sqrt(np.mean(x_query[:, idx] ** 2, axis=1)))
            chrom_weights.append(len(idx))

        chrom_scores = np.column_stack(chrom_scores)
        chrom_weights = np.asarray(chrom_weights, dtype=float)
        chrom_weights /= chrom_weights.sum()

        cnv_score = np.average(chrom_scores, axis=1, weights=chrom_weights)
        cnv_signal = np.max(np.abs(x_query), axis=1)

        cnv_adata.obs.loc[query_names, "cnv_score"] = cnv_score
        cnv_adata.obs.loc[query_names, "cnv_signal"] = cnv_signal

        if x_cnv_full is None:
            x_cnv_full = np.full((cnv_adata.n_obs, n_windows), np.nan, dtype=np.float32)

        if "cnv" not in cnv_adata.uns:
            cnv_adata.uns["cnv"] = dict(combined.uns["cnv"])

        x_cnv_full[cnv_adata.obs_names.get_indexer(query_names)] = x_query

        print(f"  median burden {np.median(cnv_score):.4f}")

    except Exception as error:
        print(f"  failed: {error}")
        failures.append({"sample": sample, "error": str(error)})

    finally:
        del target, refs, combined
        gc.collect()


In [ ]:
# Missing inferCNV results must never pass silently into downstream calls.
failures = pd.DataFrame(failures)

if not failures.empty:
    failures.to_csv(df_dir / "infercnv_failed_samples.csv", index=False)
    raise RuntimeError(f"inferCNV failed for {len(failures)} sample(s)")

assert cnv_adata.obs[["cnv_score", "cnv_signal"]].notna().all().all()

display(pd.DataFrame({"infercnv_samples_completed": [len(cnv_samples)]}))


In [ ]:
valid = cnv_adata.obs[["cnv_score", "total_counts"]].dropna()
r = np.corrcoef(valid["cnv_score"], valid["total_counts"])[0, 1]
display(pd.DataFrame({"cnv_score_vs_total_counts_r": [r]}).round(3))


In [ ]:
cnv_adata.obs.groupby("condition", observed=True)["cnv_score"].describe().round(4)


## Specimen origin and CNV-high sensitivity



The primary population definition follows specimen origin. Epithelial cells
from ER+, HER2+ and TN specimens are labelled `tumour_derived`; this does not
imply that every cell is malignant. Normal-tissue epithelial cells provide the
reference population.

`cnv_high` is a technical flag applied to every epithelial cell whose inferred
CNV burden exceeds the 95th percentile of normal epithelium. The conjunction
of tumour origin and this flag is stored as `tumour_derived_cnv_high`. This is a
conservative CNV-enriched subgroup, not a classification of every tumour-derived cell.

Released tumour-cluster barcodes are retained only as an external concordance
variable. G1-restricted summaries assess whether cell-cycle transcription
materially affects the CNV-high sensitivity analysis.


In [ ]:
condition_values = cnv_adata.obs["condition"].astype(str)
phase_values = cnv_adata.obs["phase"].astype(str)

normal_mask = condition_values.eq("N")
normal_scores = cnv_adata.obs.loc[normal_mask, "cnv_score"]
normal_g1_scores = cnv_adata.obs.loc[
    normal_mask & phase_values.eq("G1"), "cnv_score"
]

cnv_high_threshold = float(normal_scores.quantile(cnv_high_quantile))
cnv_high_threshold_g1 = float(normal_g1_scores.quantile(cnv_high_quantile))

source_tumour = pd.read_csv(
    source_tumour_file,
    dtype={"sample_code": str, "barcode": str},
)
required_columns = {"condition", "sample_code", "barcode", "source_cluster"}
assert required_columns.issubset(source_tumour.columns)


In [ ]:
assert not source_tumour.duplicated(
    ["condition", "sample_code", "barcode"]
).any()


In [ ]:
def sample_code(sample):
    sample_name = str(sample).split("_", 1)[-1]
    matches = re.findall(r"(?<!\d)(\d{4})(?!\d)", sample_name)
    return matches[-1] if matches else ""


In [ ]:
condition_values = cnv_adata.obs["condition"].astype(str)
sample_codes = cnv_adata.obs["sample"].astype(str).map(sample_code)
cell_barcodes = cnv_adata.obs_names.to_series().str.split("-GSM").str[0]

source_keys = set(source_tumour[
    ["condition", "sample_code", "barcode"]
].itertuples(index=False, name=None))
available_groups = set(source_tumour[
    ["condition", "sample_code"]
].drop_duplicates().itertuples(index=False, name=None))

cell_keys = zip(condition_values, sample_codes, cell_barcodes)


In [ ]:
condition_values = cnv_adata.obs["condition"].astype(str)
sample_codes = cnv_adata.obs["sample"].astype(str).map(sample_code)
cell_barcodes = cnv_adata.obs_names.to_series().str.split("-GSM").str[0]

source_keys = set(
    source_tumour[["condition", "sample_code", "barcode"]]
    .itertuples(index=False, name=None)
)
available_groups = set(
    source_tumour[["condition", "sample_code"]]
    .drop_duplicates()
    .itertuples(index=False, name=None)
)

cell_keys = zip(condition_values, sample_codes, cell_barcodes)
sample_keys = zip(condition_values, sample_codes)

cnv_adata.obs["source_tumour_cluster"] = np.fromiter(
    (key in source_keys for key in cell_keys),
    dtype=bool,
    count=cnv_adata.n_obs,
)
cnv_adata.obs["source_tumour_cluster_eligible"] = np.fromiter(
    (key in available_groups for key in sample_keys),
    dtype=bool,
    count=cnv_adata.n_obs,
)

transfer_exclusions_file = data_dir / "source_transfer_exclusions.csv"
transfer_exclusions = pd.read_csv(transfer_exclusions_file, dtype={"sample": str})

not_transferable = cnv_adata.obs["sample"].astype(str).isin(
    transfer_exclusions["sample"]
)
cnv_adata.obs.loc[not_transferable, "source_tumour_cluster"] = False
cnv_adata.obs.loc[not_transferable, "source_tumour_cluster_eligible"] = False

In [ ]:
cnv_adata.obs["tumour_derived"] = condition_values.ne("N")
cnv_adata.obs["cnv_high"] = cnv_adata.obs["cnv_score"].gt(
    cnv_high_threshold
)
cnv_adata.obs["tumour_derived_cnv_high"] = (
    cnv_adata.obs["tumour_derived"] & cnv_adata.obs["cnv_high"]
)
cnv_adata.obs["g1_cell"] = phase_values.eq("G1")
cnv_adata.obs["tumour_derived_cnv_high_g1"] = (
    cnv_adata.obs["tumour_derived_cnv_high"] & cnv_adata.obs["g1_cell"]
)


In [ ]:
origin_call = np.where(
    condition_values.eq("N"),
    "normal-tissue epithelium",
    "tumour-derived epithelium",
)
origin_order = ["normal-tissue epithelium", "tumour-derived epithelium"]

source_call = np.full(cnv_adata.n_obs, "not evaluated", dtype=object)
normal = condition_values.eq("N").to_numpy()
eligible = cnv_adata.obs["source_tumour_cluster_eligible"].to_numpy()
source_tumour_mask = cnv_adata.obs["source_tumour_cluster"].to_numpy()

source_call[normal] = "normal reference"
source_call[eligible] = "other epithelial"
source_call[source_tumour_mask] = "source tumour cluster"
source_call_order = [
    "normal reference", "source tumour cluster", "other epithelial",
    "not evaluated",
]


In [ ]:
cnv_adata.obs["specimen_origin"] = pd.Categorical(
    origin_call, categories=origin_order
)
cnv_adata.obs["source_tumour_cluster_call"] = pd.Categorical(
    source_call, categories=source_call_order
)

population_counts = pd.crosstab(
    cnv_adata.obs["condition"], cnv_adata.obs["specimen_origin"], margins=True
)


In [ ]:
display(population_counts)


In [ ]:
rows = []


In [ ]:
for quantile in cnv_quantiles:
    threshold = float(normal_scores.quantile(quantile))
    called = (
        cnv_adata.obs["cnv_score"].gt(threshold)
        & cnv_adata.obs["condition"].ne("N")
    )

    for condition in ["ER+", "HER2+", "TN"]:
        mask = cnv_adata.obs["condition"].eq(condition)
        rows.append({
            "normal_quantile": quantile,
            "threshold": threshold,
            "condition": condition,
            "n_cells": int(mask.sum()),
            "n_cnv_high": int(called[mask].sum()),
            "cnv_high_fraction": float(called[mask].mean()),
        })


In [ ]:
threshold_sensitivity = pd.DataFrame(rows)
threshold_sensitivity.to_csv(
    df_dir / "infercnv_threshold_sensitivity.csv", index=False
)
display(threshold_sensitivity.round(4))


In [ ]:
cycle_cnv_summary = (
    cnv_adata.obs
    .groupby(["condition", "phase"], observed=True)
    .agg(
        n_cells=("cnv_score", "size"),
        median_cnv=("cnv_score", "median"),
        cnv_high_fraction=("cnv_high", "mean"),
        tumour_derived_cnv_high_fraction=("tumour_derived_cnv_high", "mean"),
    )
    .reset_index()
)
cycle_cnv_summary["normal_all_q95"] = cnv_high_threshold
cycle_cnv_summary["normal_g1_q95"] = cnv_high_threshold_g1
cycle_cnv_summary.to_csv(
    df_dir / "infercnv_cell_cycle_sensitivity.csv", index=False
)
display(cycle_cnv_summary.round(4))


In [ ]:
sample_population_summary = (
    cnv_adata.obs
    .groupby(["sample", "condition"], observed=True)
    .agg(
        n_epithelial_cells=("tumour_derived", "size"),
        n_tumour_derived=("tumour_derived", "sum"),
        n_cnv_high=("cnv_high", "sum"),
        n_tumour_derived_cnv_high=("tumour_derived_cnv_high", "sum"),
        n_tumour_derived_cnv_high_g1=("tumour_derived_cnv_high_g1", "sum"),
        n_source_tumour_cluster=("source_tumour_cluster", "sum"),
        median_cnv=("cnv_score", "median"),
    )
    .reset_index()
)


In [ ]:
sample_population_summary["tumour_derived_cnv_high_fraction"] = (
    sample_population_summary["n_tumour_derived_cnv_high"]
    / sample_population_summary["n_epithelial_cells"]
)
sample_population_summary["tumour_derived_cnv_high_g1_fraction"] = (
    sample_population_summary["n_tumour_derived_cnv_high_g1"]
    / sample_population_summary["n_epithelial_cells"]
)
sample_population_summary["source_tumour_cluster_fraction"] = (
    sample_population_summary["n_source_tumour_cluster"]
    / sample_population_summary["n_epithelial_cells"]
)


In [ ]:
sample_population_summary.sort_values(
    ["condition", "tumour_derived_cnv_high_fraction"]
).round(3)


In [ ]:
sample_population_summary.to_csv(
    df_dir / "epithelial_population_summary_by_sample.csv", index=False
)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

sns.boxplot(
    data=cnv_adata.obs,
    x="condition",
    y="cnv_score",
    order=["N", "ER+", "HER2+", "TN"],
    showfliers=False,
    ax=ax,
)

ax.set_xlabel("")


ax.set_ylabel("inferCNV burden")
sns.despine()

save_figure(fig, "003_01_infercnv_score_by_condition")
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

sns.histplot(
    data=cnv_adata.obs,
    x="cnv_score",
    hue="condition",
    bins=100,
    stat="density",
    common_norm=False,
    element="step",
    ax=ax,
)


ax.axvline(cnv_high_threshold, linestyle="--", linewidth=1.5, color="black")
ax.set_xlabel("inferCNV burden")
ax.set_ylabel("density")
sns.despine()

save_figure(fig, "003_02_infercnv_cnv_high_threshold")
plt.show()


In [ ]:
plot_data = sample_population_summary[
    sample_population_summary["condition"].ne("N")
].copy()

fig, ax = plt.subplots(figsize=(14, 5))


sns.barplot(
    data=plot_data,
    x="sample",
    y="tumour_derived_cnv_high_fraction",
    hue="condition",
    dodge=False,
    ax=ax,
)

ax.set_xlabel("")
ax.set_ylabel("fraction of tumour-derived epithelium that is CNV-high")
ax.tick_params(axis="x", rotation=90, labelsize=7)


sns.despine()

save_figure(fig, "003_03_tumour_derived_cnv_high_fraction_by_sample")
plt.show()


In [ ]:
# transfer the inferCNV results onto the master object
assert epi.obs_names.equals(cnv_adata.obs_names)


In [ ]:
transfer_columns = [
    "cnv_score", "cnv_signal", "tumour_derived", "cnv_high",
    "tumour_derived_cnv_high", "g1_cell", "tumour_derived_cnv_high_g1",
    "specimen_origin", "source_tumour_cluster",
    "source_tumour_cluster_eligible", "source_tumour_cluster_call",
]

for column in transfer_columns:
    epi.obs[column] = cnv_adata.obs[column].reindex(epi.obs_names)


In [ ]:
boolean_columns = [
    "tumour_derived", "cnv_high", "tumour_derived_cnv_high", "g1_cell",
    "tumour_derived_cnv_high_g1", "source_tumour_cluster",
    "source_tumour_cluster_eligible",
]
for column in boolean_columns:
    epi.obs[column] = epi.obs[column].astype(bool)

epi.obs["specimen_origin"] = pd.Categorical(
    epi.obs["specimen_origin"], categories=origin_order
)
epi.obs["source_tumour_cluster_call"] = pd.Categorical(
    epi.obs["source_tumour_cluster_call"], categories=source_call_order
)


In [ ]:
epi.uns["epithelial_population_definitions"] = {
    "primary_population": "specimen origin",
    "tumour_derived": "epithelial cells from ER+, HER2+ or TN specimens",
    "normal_tissue": "epithelial cells from normal specimens",
    "tumour_derived_cnv_high": "tumour-derived and CNV burden above normal q95",
    "tumour_derived_cnv_high_limitation": (
        "conservative CNV-enriched subgroup; does not classify all tumour-derived cells"
    ),
    "cnv_method": "infercnvpy",
    "cnv_reference": "normal epithelial cells; leave-one-sample-out for normal samples",
    "gene_annotation": "GENCODE v50 joined by Ensembl gene ID",
    "expression_layer": "log-normalised expression",
    "window_size": 100,
    "step": 10,
    "dynamic_threshold": float(infercnv_dynamic_threshold),
    "cnv_score": "chromosome-level RMS burden weighted by window count",
    "cnv_high_normal_quantile": float(cnv_high_quantile),
    "cnv_high_threshold": cnv_high_threshold,
    "cnv_high_threshold_g1_reference": cnv_high_threshold_g1,
    "source_cluster_file": source_tumour_file.name,
    "source_cluster_rows": int(len(source_tumour)),
    "cnv_score_vs_total_counts_r": float(r),
}


In [ ]:
pd.DataFrame({
    "tumour_derived": [int(epi.obs["tumour_derived"].sum())],
    "normal_tissue": [int((~epi.obs["tumour_derived"]).sum())],
    "cnv_high_all_epithelium": [int(epi.obs["cnv_high"].sum())],
    "tumour_derived_cnv_high": [int(epi.obs["tumour_derived_cnv_high"].sum())],
    "tumour_derived_cnv_high_g1": [int(epi.obs["tumour_derived_cnv_high_g1"].sum())],
    "source_tumour_clusters": [int(epi.obs["source_tumour_cluster"].sum())],
})


In [ ]:
cnv_adata.obsm["X_cnv"] = x_cnv_full


In [ ]:
cnv_adata.write_h5ad(save_dir / "epithelial_infercnv.h5ad", compression="gzip")


In [ ]:
del cnv_adata, reference, x_cnv_full
gc.collect()


## scVI integration

In [ ]:
epi_scvi = epi[:, epi.var["highly_variable"]].copy()


In [ ]:
assert epi_scvi.n_obs == epi.n_obs
assert epi_scvi.n_vars == n_hvg_deviance


In [ ]:
epi_scvi.shape


In [ ]:
# sample (library) identity is the batch covariate
scvi.model.SCVI.setup_anndata(epi_scvi, layer="counts", batch_key="sample")


In [ ]:
model = scvi.model.SCVI(
    epi_scvi,
    n_latent=n_latent,
    n_layers=n_layers,
    gene_likelihood=gene_likelihood,
)


In [ ]:
# free the large intermediates; nothing below uses them
del epi_scvi
gc.collect()


In [ ]:
model


In [ ]:
# min_delta of 1.0 nat on the ELBO stops training once the remaining gain is small relative to the reconstruction term; it sets the epoch count reported
model.train(
    max_epochs=400,
    accelerator="mps",
    batch_size=1024,
    early_stopping=True,
    early_stopping_patience=20,
    early_stopping_min_delta=1.0,
)


In [ ]:
n_epochs = model.history["elbo_train"].shape[0]


In [ ]:
n_epochs


In [ ]:
model.save(str(model_dir), overwrite=True)
model.history["elbo_train"].to_csv(df_dir / "scvi_epithelial_history.csv")


In [ ]:
latent = model.get_latent_representation()
assert latent.shape == (epi.n_obs, n_latent)
assert np.isfinite(latent).all()

epi.obsm["X_scVI"] = latent
del latent, model
gc.collect()

In [ ]:
epi.obsm["X_scVI"] = latent


In [ ]:
latent.shape


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

training_history = pd.read_csv(df_dir / "scvi_epithelial_history.csv", index_col=0)
training_history.plot(ax=ax, legend=False)

ax.set_xlabel("epoch")
ax.set_ylabel("training ELBO")
sns.despine()

save_figure(fig, "003_04_scvi_epithelial_elbo")
plt.show()


## Neighbours, UMAP and Leiden clustering

In [ ]:
sc.pp.neighbors(
    epi,
    use_rep="X_scVI",
    n_neighbors=n_neighbors,
    random_state=random_seed,
)
sc.tl.umap(epi, random_state=random_seed)


In [ ]:
from sklearn.metrics import adjusted_rand_score


In [ ]:
leiden_summary = []


In [ ]:
for resolution in leiden_resolutions:
    key = f"leiden_r{resolution:g}"
    sc.tl.leiden(
        epi,
        resolution=resolution,
        key_added=key,
        flavor="igraph",
        n_iterations=2,
        directed=False,
        random_state=random_seed,
    )
    leiden_summary.append({"resolution": resolution, "n_clusters": epi.obs[key].nunique()})


In [ ]:
display(pd.DataFrame(leiden_summary))


In [ ]:
primary_key = f"leiden_r{primary_resolution:g}"
resolution_stability = pd.DataFrame({
    "resolution": leiden_resolutions,
    "ari_vs_primary": [
        adjusted_rand_score(epi.obs[primary_key], epi.obs[f"leiden_r{resolution:g}"])
        for resolution in leiden_resolutions
    ],
})
display(resolution_stability.round(3))


In [ ]:
epi.obs["epi_leiden"] = epi.obs[primary_key].copy()


In [ ]:
epi.obs["epi_leiden"].value_counts().sort_index()


## Integration diagnostics

In [ ]:
from sklearn.metrics import silhouette_score


In [ ]:
rng = np.random.default_rng(0)

n_eval = min(20000, epi.n_obs)
eval_idx = rng.choice(epi.n_obs, size=n_eval, replace=False)

z_eval = epi.obsm["X_scVI"][eval_idx]
obs_eval = epi.obs.iloc[eval_idx]

bio_asw = silhouette_score(z_eval, obs_eval["cl_annotation"], metric="euclidean")
batch_asw = silhouette_score(z_eval, obs_eval["sample"], metric="euclidean")

print(f"annotation silhouette: {bio_asw:.3f}")


In [ ]:
print(f"sample silhouette:     {batch_asw:.3f}")


In [ ]:
pd.DataFrame({
    "metric": ["annotation_silhouette", "sample_silhouette"],
    "value": [bio_asw, batch_asw],
}).to_csv(df_dir / "scvi_epithelial_integration_metrics.csv", index=False)


In [ ]:
cluster_composition = pd.crosstab(
    epi.obs["epi_leiden"], epi.obs["sample"], normalize="index"
)
cluster_composition["top_sample_fraction"] = cluster_composition.max(axis=1)
cluster_composition.to_csv(df_dir / "scvi_epithelial_cluster_sample_composition.csv")
display(cluster_composition.sort_values("top_sample_fraction", ascending=False).round(3))


In [ ]:
cluster_composition.to_csv(df_dir / "scvi_epithelial_cluster_sample_composition.csv")


In [ ]:
cluster_population = (
    epi.obs
    .groupby("epi_leiden", observed=True)
    .agg(
        n_cells=("tumour_derived", "size"),
        n_tumour_derived=("tumour_derived", "sum"),
        n_tumour_derived_cnv_high=("tumour_derived_cnv_high", "sum"),
        n_tumour_derived_cnv_high_g1=("tumour_derived_cnv_high_g1", "sum"),
        median_cnv=("cnv_score", "median"),
    )
)


In [ ]:
cluster_population["tumour_derived_fraction"] = (
    cluster_population["n_tumour_derived"] / cluster_population["n_cells"]
)
cluster_population["tumour_derived_cnv_high_fraction"] = (
    cluster_population["n_tumour_derived_cnv_high"] / cluster_population["n_cells"]
)


In [ ]:
cluster_population.sort_values(
    "tumour_derived_cnv_high_fraction", ascending=False
).round(3)


In [ ]:
cluster_population.to_csv(
    df_dir / "scvi_epithelial_cluster_population_composition.csv"
)


In [ ]:
# canonical epithelial markers, to confirm the clusters are epithelial
marker_genes = [
    g for g in ["EPCAM", "KRT8", "KRT18", "KRT19", "KRT7", "KRT5", "KRT14", "KRT17", "MUC1"]
    if g in epi.var_names
]


In [ ]:
dot = sc.pl.dotplot(
    epi,
    var_names=marker_genes,
    groupby="epi_leiden",
    layer="lognorm",
    use_raw=False,
    figsize=(6, 5),
    show=False,
)
dot.make_figure()
save_figure(fig, "003_05_epithelial_cluster_markers_dotplot")
plt.show()


## Figure 1a and related UMAPs

In [ ]:
condition_palette = CONDITION_PALETTE
condition_order = CONDITION_ORDER


In [ ]:
epi.obs["condition"] = pd.Categorical(
    epi.obs["condition"].astype(str), categories=condition_order, ordered=True
)


In [ ]:
# set the colours on the object, so plotting a subset does not have to write them
epi.uns["condition_colors"] = [condition_palette[c] for c in condition_order]


In [ ]:
epi.obs["cl_annotation"] = epi.obs["cl_annotation"].astype("category")


In [ ]:
epi.obs["sample_short"] = (
    epi.obs["sample_short"].astype("category").cat.remove_unused_categories()
)


In [ ]:
# draw N first so that the tumour conditions are not hidden underneath it
draw_order = ["N", "TN", "HER2+", "ER+"]
rank = {condition: i for i, condition in enumerate(draw_order)}

plot_idx = epi.obs["condition"].astype(str).map(rank).sort_values().index


In [ ]:
sc.pl.umap(epi, color="epi_leiden", palette="tab20b",legend_loc="on data",)


In [ ]:
sc.pl.umap(
    epi[plot_idx],
    color="condition",
    palette=condition_palette,
    frameon=True,
    show=False,
)
ax = plt.gca()
fig = ax.figure
ax.spines[["top", "right"]].set_visible(False)
ax.set_xlabel("UMAP1", fontsize=25)
ax.set_ylabel("UMAP2", fontsize=25)


ax.tick_params(axis="both", labelsize=12)

legend = ax.get_legend()


if legend is not None:
    plt.setp(legend.get_texts(), fontsize=15)
    for handle in legend.legend_handles:
        handle.set_sizes([70])

save_figure(fig, "003_06_epithelial_umap_condition")
plt.show()


In [ ]:
sample_palette = categorical_palette(epi.obs["sample_short"].cat.categories)

sc.pl.umap(
    epi, color="sample_short", palette=sample_palette,
    legend_loc=None, frameon=False, show=False,
)
sample_to_color = sample_palette

In [ ]:
# Exploratory marker ranking for epithelial clusters.
sc.tl.rank_genes_groups(
    epi,
    groupby="epi_leiden",
    layer="lognorm",
    use_raw=False,
    method="wilcoxon",
    pts=True,
    tie_correct=True,
)


In [ ]:
cluster_markers = sc.get.rank_genes_groups_df(
    epi,
    group=None,
)

cluster_markers.to_csv(
    df_dir / "epithelial_cluster_markers_wilcoxon.csv",
    index=False,
)


In [ ]:
dot = sc.pl.rank_genes_groups_dotplot(
    epi,
    groupby="epi_leiden",
    n_genes=5,
    values_to_plot="logfoldchanges",
    standard_scale="var",
    show=False,
)
dot.make_figure()
save_figure(fig, "003_07_epithelial_cluster_markers_top5")
plt.show()


In [ ]:
# samples coloured individually, with the legend grouped by condition


sc.pl.umap(epi, color="sample_short", legend_loc=None, frameon=False, show=False)

ax = plt.gca()

# read the colours scanpy just assigned, so the legend matches the plot
sample_to_color = dict(
    zip(epi.obs["sample_short"].cat.categories, epi.uns["sample_short_colors"])
)

handles = []


for condition in condition_order:

    handles.append(Line2D([], [], linestyle="none", label=condition, color="black"))

    for sample in sorted(epi.obs.loc[epi.obs["condition"] == condition, "sample_short"].unique()):
        handles.append(
            Line2D([], [], marker="o", linestyle="none", markersize=6,
                   markerfacecolor=sample_to_color[sample], markeredgecolor="none",
                   label=f"  {sample}")
        )


ax.legend(handles=handles, bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False, fontsize=7)

save_figure(fig, "003_08_epithelial_umap_sample")
plt.show()


In [ ]:
umap = epi.obsm["X_umap"]
tumour_derived_cnv_high_plot = epi.obs["tumour_derived_cnv_high"].to_numpy()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

sc.pl.umap(
    epi, color="cnv_score", color_map="viridis", vmax="p99.9",
    frameon=False, title="CNV burden", ax=axes[0], show=False,
)


axes[1].scatter(
    umap[:, 0], umap[:, 1], s=1, color="#D0D0D0",
    linewidths=0, rasterized=True,
)
axes[1].scatter(
    umap[tumour_derived_cnv_high_plot, 0], umap[tumour_derived_cnv_high_plot, 1], s=1, color="#D55E00",
    linewidths=0, rasterized=True,
)
axes[1].set_title("CNV-high tumour-derived epithelium")
axes[1].set_xticks([])
axes[1].set_yticks([])
axes[1].set_aspect("equal", adjustable="datalim")


axes[1].set_frame_on(False)
axes[1].legend(
    handles=[
        Line2D([], [], marker="o", linestyle="none", color="#D0D0D0", label="other epithelium"),
        Line2D([], [], marker="o", linestyle="none", color="#D55E00", label="CNV-high tumour-derived"),
    ],
    frameon=False, loc="lower left", markerscale=3, fontsize=8,
)


sc.pl.umap(
    epi, color="epi_leiden", legend_loc="right margin",
    frameon=False, title="Epithelial clusters", ax=axes[2], show=False,
)

fig.tight_layout()
fig.savefig(
    fig_dir / "epithelial_umap_population_and_cnv",
    dpi=300,
    bbox_inches="tight",
)
plt.show()


In [ ]:
from cap.publication_figure_state import save_cnv_panels

save_cnv_panels(epi, fig_dir)

In [ ]:
sc.pl.umap(epi, color='cnv_score', color_map='viridis', frameon=False, show=False, vmax='p99.9')


In [ ]:
sc.pl.umap(epi, color='cnv_score', color_map='RdYlBu_r', frameon=False, show=False, vmax='p99.9')


In [ ]:
origin_palette = {
    "normal-tissue epithelium": "#56B4E9",
    "tumour-derived epithelium": "#D55E00",
}


In [ ]:
sc.pl.umap(
    epi,
    color="specimen_origin",
    palette=origin_palette,
    frameon=False,
    show=False,
    title="Epithelial cells by specimen origin",
)
fig = plt.gcf()  #capture this plot
save_figure(fig, "003_09_epithelial_umap_specimen_origin")
plt.show()


In [ ]:
checkpoint_dir = project / "Adata_Saves" / "checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

checkpoint_path = checkpoint_dir / "epi_primary_scvi_umap_leiden.h5ad"
tmp_path = checkpoint_path.with_name("epi_primary_scvi_umap_leiden.tmp.h5ad")

display_path(checkpoint_path, root=project)
epi.write_h5ad(tmp_path, compression="gzip")
os.replace(tmp_path, checkpoint_path)

## Feature-set sensitivity


The primary scVI model uses only the deviance-ranked genes. A second model adds
the antigen and meta-programme genes to test whether that targeted feature choice changes
the neighbourhood or cluster structure beyond ordinary run-to-run variation.


In [ ]:
epi_protected = epi[:, epi.var["highly_variable_protected"]].copy()
assert epi_protected.n_vars == n_hvg_protected


In [ ]:
scvi.model.SCVI.setup_anndata(
    epi_protected, layer="counts", batch_key="sample"
)


In [ ]:
model_protected = scvi.model.SCVI(
    epi_protected,
    n_latent=n_latent,
    n_layers=n_layers,
    gene_likelihood=gene_likelihood,
)


In [ ]:
model_protected.train(
    max_epochs=400,
    accelerator="mps",
    batch_size=1024,
    early_stopping=True,
    early_stopping_patience=20,
    early_stopping_min_delta=1.0,
)


In [ ]:
epi.obsm["X_scVI_protected"] = model_protected.get_latent_representation()


In [ ]:
del model_protected
gc.collect()


In [ ]:
sc.pp.neighbors(
    epi,
    use_rep="X_scVI_protected",
    n_neighbors=n_neighbors,
    random_state=random_seed,
    key_added="protected_neighbors",
)


In [ ]:
sc.tl.leiden(
    epi,
    resolution=primary_resolution,
    key_added="epi_leiden_protected",
    neighbors_key="protected_neighbors",
    flavor="igraph",
    n_iterations=2,
    directed=False,
    random_state=random_seed,
)

del epi_protected


In [ ]:
gc.collect()


In [ ]:
epi_seed = epi[:, epi.var["highly_variable"]].copy()
scvi.settings.seed = random_seed + 1
scvi.model.SCVI.setup_anndata(epi_seed, layer="counts", batch_key="sample")
 


In [ ]:
model_seed = scvi.model.SCVI(
    epi_seed,
    n_latent=n_latent,
    n_layers=n_layers,
    gene_likelihood=gene_likelihood,
)


In [ ]:
model_seed.train(
    max_epochs=400,
    accelerator="mps",
    batch_size=1024,
    early_stopping=True,
    early_stopping_patience=20,
    early_stopping_min_delta=1.0,
)

epi.obsm["X_scVI_seed2"] = model_seed.get_latent_representation()
scvi.settings.seed = random_seed


In [ ]:
sc.pp.neighbors(
    epi,
    use_rep="X_scVI_seed2",
    n_neighbors=n_neighbors,
    random_state=random_seed,
    key_added="seed2_neighbors",
)


In [ ]:
sc.tl.leiden(
    epi,
    resolution=primary_resolution,
    key_added="epi_leiden_seed2",
    neighbors_key="seed2_neighbors",
    flavor="igraph",
    n_iterations=2,
    directed=False,
    random_state=random_seed,
)

del epi_seed, model_seed


In [ ]:
gc.collect()


In [ ]:
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.neighbors import NearestNeighbors


In [ ]:
k_overlap = n_neighbors
overlap_idx = eval_idx


In [ ]:
def knn_indices(representation):
    values = epi.obsm[representation][overlap_idx]
    model_nn = NearestNeighbors(n_neighbors=k_overlap + 1).fit(values)
    return model_nn.kneighbors(values, return_distance=False)[:, 1:]


In [ ]:
primary_neighbors = knn_indices("X_scVI")
comparisons = [
    ("protected feature set", "X_scVI_protected", "epi_leiden_protected"),
    ("different seed", "X_scVI_seed2", "epi_leiden_seed2"),
]

rows = []


In [ ]:
for label, representation, clustering in comparisons:
    other_neighbors = knn_indices(representation)
    overlap = np.array([
        len(set(left) & set(right)) / k_overlap
        for left, right in zip(primary_neighbors, other_neighbors)
    ])
    rows.append({
        "comparison": label,
        "leiden_ari": adjusted_rand_score(epi.obs["epi_leiden"], epi.obs[clustering]),
        "leiden_nmi": normalized_mutual_info_score(epi.obs["epi_leiden"], epi.obs[clustering]),
        "knn_overlap_mean": overlap.mean(),
        "knn_overlap_median": np.median(overlap),
        "n_clusters": epi.obs[clustering].nunique(),
    })


In [ ]:
latent_comparison = pd.DataFrame(rows)
latent_comparison.to_csv(df_dir / "scvi_latent_comparison.csv", index=False)
display(latent_comparison.round(3))


In [ ]:
rows = []


In [ ]:
for label, representation in [
    ("primary_deviance", "X_scVI"),
    ("protected_sensitivity", "X_scVI_protected"),
    ("primary_second_seed", "X_scVI_seed2"),
]:
    values = epi.obsm[representation][eval_idx]
    rows.append({
        "model": label,
        "annotation_silhouette": silhouette_score(values, obs_eval["cl_annotation"]),
        "sample_silhouette": silhouette_score(values, obs_eval["sample"]),
        "condition_silhouette": silhouette_score(values, obs_eval["condition"].astype(str)),
    })


In [ ]:
model_comparison = pd.DataFrame(rows)
display(model_comparison.round(3))


In [ ]:
# free the large intermediates; nothing below uses them
del model
gc.collect()


In [ ]:
rows = []


In [ ]:
for annotation in obs_eval["cl_annotation"].value_counts().index:
    mask = obs_eval["cl_annotation"].eq(annotation).to_numpy()
    if mask.sum() < 100 or obs_eval.loc[mask, "sample"].nunique() < 2:
        continue

    row = {"cl_annotation": annotation, "n_cells": int(mask.sum())}
    for label, representation in [
        ("primary_deviance", "X_scVI"),
        ("protected_sensitivity", "X_scVI_protected"),
        ("primary_second_seed", "X_scVI_seed2"),
    ]:
        row[label] = silhouette_score(
            epi.obsm[representation][eval_idx][mask],
            obs_eval.loc[mask, "sample"],
        )
    rows.append(row)


In [ ]:
within_annotation = pd.DataFrame(rows)
display(within_annotation.round(3))


In [ ]:
cluster_overlap = pd.crosstab(
    epi.obs["epi_leiden"], epi.obs["epi_leiden_protected"]
)
best_match = cluster_overlap.max(axis=1) / cluster_overlap.sum(axis=1)

feature_effect = latent_comparison.iloc[0]
seed_floor = latent_comparison.iloc[1]


In [ ]:
epi.uns["protected_gene_sensitivity"] = {
    "primary_features": int(n_hvg_deviance),
    "protected_features": int(n_hvg_protected),
    "protected_added": int(n_protected_added),
    "feature_set_ari": float(feature_effect["leiden_ari"]),
    "seed_floor_ari": float(seed_floor["leiden_ari"]),
    "feature_set_knn_overlap": float(feature_effect["knn_overlap_mean"]),
    "seed_floor_knn_overlap": float(seed_floor["knn_overlap_mean"]),
    "k": int(k_overlap),
}

model_comparison.to_csv(df_dir / "scvi_feature_sensitivity.csv", index=False)


In [ ]:
within_annotation.to_csv(
    df_dir / "scvi_feature_sensitivity_by_annotation.csv", index=False
);


In [ ]:
best_match.rename("largest_matching_fraction").to_csv(
    df_dir / "scvi_feature_cluster_correspondence.csv"
)


## Save

In [ ]:
final_positive = {
    gene: int((np.asarray(epi[:, gene].layers["counts"].todense()).ravel() > 0).sum())
    for gene in target_genes
}

assert epi.n_obs == n_cells_start
assert epi.n_vars == n_genes_start
assert final_positive == {gene: baseline_positive[gene] for gene in final_positive}


In [ ]:
epi.uns["epithelial_integration"] = {
    "hvg_method": "binomial deviance (scry), maximum across samples",
    "primary_feature_set": "deviance-ranked genes only",
    "n_hvg_primary": int(n_hvg_deviance),
    "n_hvg_protected_sensitivity": int(n_hvg_protected),
    "scvi_n_latent": int(n_latent),
    "scvi_n_layers": int(n_layers),
    "scvi_gene_likelihood": gene_likelihood,
    "scvi_batch_key": "sample",
    "scvi_epochs": int(n_epochs),
    "n_neighbors": int(n_neighbors),
    "leiden_resolutions": [float(value) for value in leiden_resolutions],
    "leiden_primary": float(primary_resolution),
    "seed": int(random_seed),
}


Source barcode transfer audit. Recovery after QC is reported separately from sample identity; ER-MH0001 is excluded from transfer. This is optional concordance, not the primary specimen-origin definition.

In [ ]:
from cap.paths import results_dir, notebook_dir as get_notebook_dir
from cap.publication_targeted_checks import barcode_audit

check_dir = project / "results" / "publication_checks"
check_dir.mkdir(parents=True, exist_ok=True)

# Use project if this is the data/reference root expected by barcode_audit.
repository = project
barcode_recovery = barcode_audit(epi, project, repository, check_dir)

available = barcode_recovery["source_group_available"].fillna(False)
display(barcode_recovery.loc[available])

In [ ]:
out_path = save_dir / "epithelial_tumour_scvi.h5ad"


In [ ]:
for name in list(epi.layers.keys()):
    if not sp.issparse(epi.layers[name]):
        epi.layers[name] = sp.csr_matrix(epi.layers[name])


In [ ]:
if not sp.issparse(epi.X):
    epi.X = sp.csr_matrix(epi.X)


In [ ]:
if not epi.uns.get("figure_only", False):
    # Write safely before replacing the existing file.
    tmp_path = out_path.with_suffix(".h5ad.tmp")
    epi.write_h5ad(tmp_path, compression="gzip")
    os.replace(tmp_path, out_path)

    display(pd.DataFrame({"saved": [out_path.name]}))

In [ ]:
check = sc.read_h5ad(out_path, backed="r")

assert check.n_obs == epi.n_obs
assert check.n_vars == epi.n_vars


In [ ]:
check.shape


In [ ]:
del check
gc.collect()


In [ ]:
import session_info


In [ ]:
session_info.show()
